In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\choud\Downloads\wiki_hybrid_chunks_300.csv")
print(df.columns)
print(df.head(3))


Index(['group_id', 'section', 'chunk_id', 'chunk_text'], dtype='object')
                           group_id   section    chunk_id  \
0  00163fe7688e71ce06f495a6811fef71  FullText  FullText_1   
1  00163fe7688e71ce06f495a6811fef71  FullText  FullText_2   
2  00163fe7688e71ce06f495a6811fef71  FullText  FullText_3   

                                          chunk_text  
0  marieve herington born february 22, 1988 is a ...  
1  and members of the royal jelly orchestra, appe...  
2  relentless there s that word again optimism of...  


In [10]:
import pandas as pd
import re

# Load your dataset
df = pd.read_csv(r"C:\Users\choud\Downloads\wiki_hybrid_chunks_300.csv")

def clean_text(text):
    # Remove common date/citation patterns like 'Retrieved October 8, 2015'
    text = re.sub(r'\b(retrieved|retrieval)\b.*?(\d{4}|\d{1,2}, \d{4})', '', text, flags=re.IGNORECASE)
    # Remove dates like '8 October 2015' or 'October 8, 2015'
    text = re.sub(r'\b\d{1,2} [A-Za-z]+ \d{4}\b', '', text)
    text = re.sub(r'\b[A-Za-z]+ \d{1,2}, \d{4}\b', '', text)
    # Remove bracketed references like [12]
    text = re.sub(r'\[[^\]]*\]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_chunk_useful(text):
    # Filter out very short chunks or chunks with lots of numbers (likely references)
    if len(text) < 30:
        return False
    if len(re.findall(r'\d', text)) > 10:
        return False
    return True

def generate_question(text):
    text = clean_text(text)
    if not is_chunk_useful(text):
        return None
    if " is " in text:
        return "What is " + text.split(" is ")[0].strip() + "?"
    elif " was " in text:
        return "Who was " + text.split(" was ")[0].strip() + "?"
    elif " are " in text:
        return "What are " + text.split(" are ")[0].strip() + "?"
    elif " refers to " in text:
        return text.split(" refers to ")[0].strip() + " refers to what?"
    else:
        # fallback generic question
        return "Tell me something about " + text[:50].strip() + "..."

# Generate questions from up to 150 chunks
n = min(150, len(df))
sampled_rows = df.sample(n=n, replace=False, random_state=42)

questions = []
for _, row in sampled_rows.iterrows():
    q = generate_question(row['chunk_text'])
    if q:
        questions.append(q)

# Remove duplicates and empty questions
questions = list(dict.fromkeys([q for q in questions if q]))

print(f"Generated {len(questions)} unique, cleaned questions:\n")
for i, question in enumerate(questions, 1):
    print(f"{i}. {question}")


Generated 25 unique, cleaned questions:

1. What is . . zoller seitz, matt . american horror story hotel?
2. Who was on , the european premiére of spandau ballet s soul boys of the western world and four james bond royal world premiéres die another day on attended by queen elizabeth ii and prince philip , skyfal on attended by charles, prince of wales and camilla, duchess of cornwall , 7 spectre on attended by prince william, duke of cambridge and catherine, duchess of cambridge 8 and no time to die on attended by charles, camilla, william, catherine. the hall held its first 3d world premiére of titanic 3d, on , with james cameron and kate winslet in attendance. l since 2009, the hall has also curated regular seasons of english language film and live orchestra screenings, including the lord of the rings trilogy, gladiator, star trek, star trek into darkness, interstellar, the matrix, west side story, breakfast at tiffany s, back to the future, jaws, the harry potter series, black panth

In [11]:
import pandas as pd
import re

df = pd.read_csv(r"C:\Users\choud\Downloads\wiki_hybrid_chunks_300.csv")

def clean_text(text):
    text = re.sub(r'\b(retrieved|retrieval)\b.*?(\d{4}|\d{1,2}, \d{4})', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\d{1,2} [A-Za-z]+ \d{4}\b', '', text)
    text = re.sub(r'\b[A-Za-z]+ \d{1,2}, \d{4}\b', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_chunk_useful(text):
    if len(text) < 40:
        return False
    punctuation_count = sum(text.count(c) for c in [".", ",", "[", "]", "(", ")", "-", "_"])
    if punctuation_count / len(text) > 0.1:
        return False
    bad_words = ['archived', 'retrieved', 'original', 'citation', 'source']
    if any(word in text.lower() for word in bad_words):
        return False
    if len(re.findall(r'\d', text)) > 5:
        return False
    return True

def generate_question(text):
    text = clean_text(text)
    if not is_chunk_useful(text):
        return None
    if " is " in text:
        return "What is " + text.split(" is ")[0].strip() + "?"
    elif " was " in text:
        return "Who was " + text.split(" was ")[0].strip() + "?"
    elif " are " in text:
        return "What are " + text.split(" are ")[0].strip() + "?"
    elif " refers to " in text:
        return text.split(" refers to ")[0].strip() + " refers to what?"
    else:
        return "Tell me something about " + text[:50].strip() + "..."

n = min(200, len(df))
sampled_rows = df.sample(n=n, replace=False, random_state=42)

questions = []
for _, row in sampled_rows.iterrows():
    q = generate_question(row['chunk_text'])
    if q:
        questions.append(q)

questions = list(dict.fromkeys(questions))

print(f"Generated {len(questions)} unique, cleaner questions:\n")
for i, question in enumerate(questions, 1):
    print(f"{i}. {question}")


Generated 14 unique, cleaner questions:

1. What is that no one ever expected. rocky completely dominates the first round, leaving lang enraged and bewildered after the bell. lang gains the upper hand in the second round, and rocky adopts an entirely different strategy that angers and confuses apollo by intentionally taking a beating from lang, even getting knocked down twice, all the while taunting lang that he cannot knock him out. by the third round, lang, who?
2. What is the forbidden zone use their psychic powers to deter the apes, ursus and his gorillas are frightened, but dr zaius becomes enraged, charging forward. ursus and his soldiers then follow zaius, finding the entrance to the mutant city and leading the apes into the heart of the mutant world which houses a nuclear weapon. ursus meets the mutant leader, who?
3. Tell me something about people of spanish descent english radio actresses...
4. What is with gloria. however, melman the giraffe shows up and divulges romantic fe